<a href="https://colab.research.google.com/github/InduSajith/ai_portfolio/blob/main/A3C_for_Kung_Fu.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A3C for Kung Fu

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [ ]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!pip install ale-py
!apt-get install -y swig
!pip install gymnasium[box2d]

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
swig is already the newest version (4.2.0-2ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.


### Importing the libraries

In [ ]:
import cv2
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.multiprocessing as mp
import torch.distributions as distributions
from torch.distributions import Categorical
import ale_py
import gymnasium as gym
from gymnasium.spaces import Box
from gymnasium import ObservationWrapper

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [ ]:
class Network(nn.Module):

  # The constructor method
  def __init__(self,action_size):

    # calls and initializes the parent class within the custom child class
    super(Network,self).__init__()
    self.conv1 = torch.nn.Conv2d(in_channels=4, out_channels=32,kernel_size=(3,3),stride=2)  # Assigns the first convolutional layer to this specific instance
    self.conv2 = torch.nn.Conv2d(in_channels=32, out_channels=32,kernel_size=(3,3),stride=2) # Assigns the second convolutional layer to this specific instance
    self.conv3 = torch.nn.Conv2d(in_channels=32, out_channels=32,kernel_size=(3,3),stride=2)
    self.flatten = torch.nn.Flatten() # flattening layer
    self.fc1 = torch.nn.Linear(512, 128) # first fully connected layer
    self.fc2a = torch.nn.Linear(128, action_size) # second fully connected layer for the action
    self.fc2s = torch.nn.Linear(128, 1) # second fully connected layer for the state value

# forward method to forward propogate the input image from the input layer to the output layer
  def forward(self,state):
    x = self.conv1(state) # forward propogating the signal to the first convolutional layer
    x = F.relu(x)         # applying Rectifier Activation function ReLU to activate the signal
    x = self.conv2(x)     # forward propogating the signal to the second convolutional layer
    x = F.relu(x)
    x = self.conv3(x)
    x = F.relu(x)

    x = self.flatten(x)   # forward propogating the signal to the flattening layer

    x = self.fc1(x)       # forward propogating the signal to the first fully connected layer
    x = F.relu(x)         # applying Rectifier Activation function ReLU to activate the signal
    action_values = self.fc2a(x) # forward propogating the signal to the to the final output layer for the action
    state_value = self.fc2s(x)[0]   # forward propogating the signal to the second final output layer for the state value
    return action_values, state_value






## Part 2 - Training the AI

### Setting up the environment

In [ ]:
class PreprocessAtari(ObservationWrapper):

  def __init__(self, env, height = 42, width = 42, crop = lambda img: img, dim_order = 'pytorch', color = False, n_frames = 4):
    super(PreprocessAtari, self).__init__(env)
    self.img_size = (height, width)
    self.crop = crop
    self.dim_order = dim_order
    self.color = color
    self.frame_stack = n_frames
    n_channels = 3 * n_frames if color else n_frames
    obs_shape = {'tensorflow': (height, width, n_channels), 'pytorch': (n_channels, height, width)}[dim_order]
    self.observation_space = Box(0.0, 1.0, obs_shape)
    self.frames = np.zeros(obs_shape, dtype = np.float32)

  def reset(self):
    self.frames = np.zeros_like(self.frames)
    obs, info = self.env.reset()
    self.update_buffer(obs)
    return self.frames, info

  def observation(self, img):
    img = self.crop(img)
    img = cv2.resize(img, self.img_size)
    if not self.color:
      if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype('float32') / 255.
    if self.color:
      self.frames = np.roll(self.frames, shift = -3, axis = 0)
    else:
      self.frames = np.roll(self.frames, shift = -1, axis = 0)
    if self.color:
      self.frames[-3:] = img
    else:
      self.frames[-1] = img
    return self.frames

  def update_buffer(self, obs):
    self.frames = self.observation(obs)

def make_env():
  env = gym.make('KungFuMasterNoFrameskip-v0', render_mode = 'rgb_array') # The KungFuMaster environment was renamed 'KungFuMasterNoFrameskip-v0'
  env = PreprocessAtari(env, height = 42, width = 42, crop = lambda img: img, dim_order = 'pytorch', color = False, n_frames = 4)
  return env

env = make_env()

state_shape = env.observation_space.shape
number_actions = env.action_space.n
print("State shape:", state_shape)
print("Number actions:", number_actions)
print("Action names:", env.env.env.env.get_action_meanings())

State shape: (4, 42, 42)
Number actions: 14
Action names: ['NOOP', 'UP', 'RIGHT', 'LEFT', 'DOWN', 'DOWNRIGHT', 'DOWNLEFT', 'RIGHTFIRE', 'LEFTFIRE', 'DOWNFIRE', 'UPRIGHTFIRE', 'UPLEFTFIRE', 'DOWNRIGHTFIRE', 'DOWNLEFTFIRE']


### Initializing the hyperparameters

In [ ]:
learning_rate = 1e-4
discount_factor = 0.99
number_environments = 10 # Multiple agents are trained in multiple environments in parallel in A3C

### Implementing the A3C class

In [ ]:
class Agent():

  def __init__(self,action_size): # Constructor method to define the object variables (object being the Agent)
    self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") # Automatically detect if an NVIDIA GPU with CUDA is available
    self.action_size = action_size
    self.network = Network(action_size).to(self.device) # Creating the brain of the model
    self.optimizer = torch.optim.Adam(self.network.parameters(),lr = learning_rate)  # Tool to update the weights of the network while backpropogating the loss

  def act(self, state): # Method for agent to play an action given a certain state
    if state.ndim == 3: # Checking whether the state is a single observation and is not in a batch
       state = [state] # Adding extra pair of brackets(extra dimension of the batch) to the numpy array to make sure the state is now in a batch
    state = torch.tensor(state, dtype = torch.float32, device = self.device) # Converts the numpy array to torch tensor
    action_values, _ = self.network(state) # Pytorch shortcut to call the forward method of network class
    policy = F.softmax(action_values, dim=-1) # softmax function converts the action_values into probabilities and they are stored into a policy tensor
    return np.array([np.random.choice(len(p), p = p) for p in policy.detach().cpu().numpy()]) # Policy tensor(action values) is converted into a numpy array

  def step(self, state, action, reward, next_state, done):
    batch_size = state.shape[0]
    state = torch.tensor(state, dtype = torch.float32, device = self.device)
    next_state = torch.tensor(next_state, dtype = torch.float32, device = self.device)
    reward =  torch.tensor(reward, dtype = torch.float32, device = self.device)
    done =  torch.tensor(done, dtype = torch.bool, device = self.device).to(dtype = torch.float32)
    action_values, state_value = self.network(state)
    _, next_state_value = self.network(next_state)
    target_state_value = reward + discount_factor * next_state_value * (1 - done) # Bellman equation
    advantage = target_state_value - state_value
    probs = F.softmax(action_values, dim = -1)
    logprobs = F.log_softmax(action_values, dim = -1)
    entropy = -torch.sum(probs * logprobs, axis = -1)
    batch_idx = np.arange(batch_size)
    logp_actions = logprobs[batch_idx, action]
    actor_loss = -(logp_actions * advantage.detach()).mean()-0.001 * entropy.mean()
    critic_loss = F.mse_loss(target_state_value.detach(), state_value) # mean square error loss between the target state value and the state value
    total_loss = actor_loss + critic_loss
    self.optimizer.zero_grad() # Reset the optimizer by zeroing out the gradience to prevent accumalations from the previous iterations
    total_loss.backward() # Back propogates the total loss
    self.optimizer.step() # Updates the weights of the neural network to minimize the total loss
    return actor_loss.item(), critic_loss.item()





### Initializing the A3C agent

In [ ]:
agent = Agent(number_actions)


### Evaluating our A3C agent on a certain number of episodes

In [ ]:
def evaluate(agent, env, n_episodes = 1): # Returns the list of total rewards on the ceratin number of episodes on which the A3C is evaluated
  episodes_rewards = []
  for _ in range(n_episodes):
    state, _ = env.reset() # Reset method initializes the state
    total_reward = 0
    while True:
      action = agent.act(state)
      state, reward, done, info, _ = env.step(action[0])
      total_reward += reward
      if done:
        break
    episodes_rewards.append(total_reward)
  return episodes_rewards


### Managing multiple environments simultaneously

In [ ]:
class EnvBatch: # Helps to handle multiple environments simultaneously (parallelized reinforcement learning)

  def __init__(self, n_envs = 10):
    self.envs = [make_env() for _ in range(n_envs)] # Object variable to hold the list of multiple environments

  def reset(self): # Method to reset the multiple environments at the same time
    _states = [] # List variable to reset the initialized states of each of the environments
    for env in self.envs:
      _states.append(env.reset()[0]) # Adds the initialized state of each environment in to a list
    return np.array(_states) # Returns the initialized state list in the form of numpy array

  def step(self, actions):
    next_states, rewards, dones, infos, _ = map(np.array, zip(*[env.step(a) for env, a in zip(self.envs, actions)]))
    for i in range(len(self.envs)):
      if dones[i]: # Checking whether the done booleans is true in the specific environment
        next_states[i] = self.envs[i].reset()[0] # Resets the state of the specific environment
    return next_states, rewards, dones, infos








### Training the A3C agent

In [ ]:
import tqdm

env_batch = EnvBatch(number_environments)
batch_states = env_batch.reset()

with tqdm.trange(0, 3001) as progress_bar:
  for i in progress_bar:
    batch_actions = agent.act(batch_states)
    batch_next_states, batch_rewards, batch_dones, _ = env_batch.step(batch_actions)
    batch_rewards *= 0.01
    agent.step(batch_states, batch_actions, batch_rewards, batch_next_states, batch_dones)
    batch_states = batch_next_states
    if i % 1000 == 0:
      print("Average agent reward: ", np.mean(evaluate(agent, env, n_episodes = 10)))

  0%|          | 0/3001 [00:00<?, ?it/s]/tmp/ipykernel_3545/766786629.py:33: UserWarning: Using a target size (torch.Size([1])) that is different to the input size (torch.Size([10])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  critic_loss = F.mse_loss(target_state_value.detach(), state_value) # mean square error loss between the target state value and the state value
/tmp/ipykernel_3545/766786629.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  state = torch.tensor(state, dtype = torch.float32, device = self.device) # Converts the numpy array to torch tensor
  0%|          | 13/3001 [02:03<5:40:18,  6.83s/it]  

Average agent reward:  680.0


 34%|███▎      | 1010/3001 [04:18<1:55:12,  3.47s/it]

Average agent reward:  1150.0


 67%|██████▋   | 2009/3001 [06:26<42:06,  2.55s/it]

Average agent reward:  1050.0


100%|██████████| 3001/3001 [08:35<00:00,  5.82it/s]

Average agent reward:  980.0


## Part 3 - Visualizing the results

In [ ]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent, env):
  state, _ = env.reset()
  done = False
  frames = []
  while not done:
    frame = env.render()
    frames.append(frame)
    action = agent.act(state)
    state, reward, done, _, _ = env.step(action[0])
  env.close()
  imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent, env)

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()